## The scheduling problem — what `kube-scheduler` does

When you `kubectl apply` a Deployment, controllers create the ReplicaSet and Pods, and the API server writes them to `etcd` with `spec.nodeName` **empty** — the pods are `Pending`. No node is chosen yet.

The **`kube-scheduler`** takes over with a tight loop:

1. **Watch** the API server for pods with no `nodeName`.
2. **Filter** the nodes — keep only those that *could* run the pod (enough free CPU/memory, taints tolerated, node selectors satisfied, volumes available in the right zone).
3. **Score** the survivors — 0–100 for how good a fit each is.
4. **Bind** — write `spec.nodeName` on the best node via the API server's `bind` subresource.
5. The **kubelet** on that node sees its own name on the pod, pulls the image, starts the container.

```
API server (Pods, nodeName="") --watch--> kube-scheduler
   filter → score → bind          --PATCH spec.nodeName-->  API server
                                                 ↑ kubelet sees its name, starts it
```

Three things to internalise:

- **The scheduler runs nothing.** It only *decides*; the kubelet runs it.
- **It's a controller** — the same read-act-write loop as everything else.
- **It's pluggable** — a cluster can run multiple schedulers or replace the default. Most don't.

The rest of the module is the **inputs** that bias this decision: how much you ask for, where you'll fit, where you won't, and where you want to spread. On our map this is the **kube-scheduler** component in the control plane — the target every Scheduling chip points at.